# cellPLM — Cell-Type Annotation Wrapper (MS dataset)

Verbatim recipe from `models/cellPLM/tutorials/cell_type_annotation.ipynb`:
`CellTypeAnnotationPipeline` with `PRETRAIN_VERSION='20230926_85M'`, `set_seed(42)`,
default pipeline + model configs. Only override: `model_config['out_dim'] = n_classes`.

Adaptations: at the end, slice predicted AnnData to `split == 'test'` and write
`ms_annotation.h5ad` with `obs_names` aligned to `filtered_ms_adata.h5ad`.

**Deviation from tutorial:** the upstream tutorial declares `DEVICE = 'cuda:3'` at
module level but never passes it to `pipeline.fit()` / `pipeline.predict()`, both of
which default to `device='cpu'`. We pass `device=DEVICE` explicitly so training runs
on GPU as the paper intends; running on CPU is infeasible at our patience budget.

In [1]:
import sys, os
sys.path.insert(0, "/data/benchmark/models/cellPLM")

import warnings
warnings.filterwarnings("ignore")

import hdf5plugin
import numpy as np
import pandas as pd
import anndata as ad
from scipy.sparse import csr_matrix
from sklearn.metrics import accuracy_score, f1_score
from CellPLM.utils import set_seed
from CellPLM.pipeline.cell_type_annotation import (
    CellTypeAnnotationPipeline,
    CellTypeAnnotationDefaultPipelineConfig,
    CellTypeAnnotationDefaultModelConfig,
)

In [2]:
PRETRAIN_VERSION = "20230926_85M"   # exact version used in the cellPLM CT tutorial
DEVICE = "cuda:0"
set_seed(42)                         # exact seed used in the tutorial

In [3]:
# --- Load MS dataset (mirrors the cellPLM tutorial's MS branch) ---
data_train = ad.read_h5ad("/data/benchmark/data/cellPLM/data/c_data.h5ad")
data_test = ad.read_h5ad("/data/benchmark/data/cellPLM/data/filtered_ms_adata.h5ad")

# Both files carry an `index_column` var entry per the cellPLM tutorial.
data_train.var = data_train.var.set_index("index_column")
data_test.var = data_test.var.set_index("index_column")

# Save the test obs_names before concatenation (we'll need these for the output file).
test_obs_names = data_test.obs_names.tolist()

train_num = data_train.shape[0]
data = ad.concat([data_train, data_test])
data.var_names_make_unique()

# The MS files store the label under `Factor Value[inferred cell type - authors labels]`.
# CellPLM expects `obs['celltype']` — normalize.
if "celltype" not in data.obs.columns:
    data.obs["celltype"] = data.obs["Factor Value[inferred cell type - authors labels]"].astype("category")

# Split: 90% of training cells = train, 10% = valid, all test cells = test.
# (Tutorial uses chained-indexing assignment; we use array assignment to avoid
# SettingWithCopyWarning / pandas 2.x silent-no-op behavior.)
split_array = np.array(["test"] * len(data), dtype=object)
tr = np.random.permutation(train_num)
split_array[tr[:int(train_num * 0.9)]] = "train"
split_array[tr[int(train_num * 0.9):train_num]] = "valid"
data.obs["split"] = split_array

In [4]:
# --- Configs (verbatim; only override out_dim) ---
pipeline_config = CellTypeAnnotationDefaultPipelineConfig.copy()
model_config = CellTypeAnnotationDefaultModelConfig.copy()
model_config["out_dim"] = data.obs["celltype"].nunique()
print("pipeline_config:", pipeline_config)
print("model_config:", model_config)

pipeline_config: {'es': 200, 'lr': 0.005, 'wd': 1e-07, 'scheduler': 'plat', 'epochs': 2000, 'max_eval_batch_size': 100000, 'hvg': 3000, 'patience': 25, 'workers': 0}
model_config: {'drop_node_rate': 0.3, 'dec_layers': 1, 'model_dropout': 0.5, 'mask_node_rate': 0.75, 'mask_feature_rate': 0.25, 'dec_mod': 'mlp', 'latent_mod': 'ae', 'head_type': 'annotation', 'max_batch_size': 70000, 'out_dim': 18}


In [5]:
# --- Fit + predict (canonical pipeline API) ---
pipeline = CellTypeAnnotationPipeline(
    pretrain_prefix=PRETRAIN_VERSION,
    overwrite_config=model_config,
    pretrain_directory="/data/benchmark/data/cellPLM/ckpt",
)

pipeline.fit(
    data,
    pipeline_config,
    split_field="split",
    train_split="train",
    valid_split="valid",
    label_fields=["celltype"],
    device=DEVICE,
)

predictions = pipeline.predict(data, pipeline_config, device=DEVICE)
# `predictions` is a 1-D array of label indices over the full `data` (train+valid+test).

After filtering, 2763 genes remain.


  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 1/2000 [00:01<40:46,  1.22s/it]

Epoch 0 | Train loss: 2.9089 | Valid loss: 2.8298
Train ACC: 0.0461 | Valid ACC: 0.0625 | Train f1: 0.0261 | Valid f1: 0.0241 | Train pre: 0.0418 | Valid pre: 0.0149


  0%|          | 2/2000 [00:01<25:47,  1.29it/s]

...

Epoch 395 | Train loss: 0.0214 | Valid loss: 0.7957
Train ACC: 0.9943 | Valid ACC: 0.8102 | Train f1: 0.9948 | Valid f1: 0.8122 | Train pre: 0.9955 | Valid pre: 0.8473


 20%|█▉        | 397/2000 [02:47<11:32,  2.32it/s]

Epoch 396 | Train loss: 0.0273 | Valid loss: 0.8008
Train ACC: 0.9958 | Valid ACC: 0.8089 | Train f1: 0.9939 | Valid f1: 0.8100 | Train pre: 0.9921 | Valid pre: 0.8447


 20%|█▉        | 397/2000 [02:47<11:18,  2.36it/s]

Epoch 397 | Train loss: 0.0210 | Valid loss: 0.7877
Train ACC: 0.9969 | Valid ACC: 0.8082 | Train f1: 0.9966 | Valid f1: 0.8095 | Train pre: 0.9962 | Valid pre: 0.8442
Early stopped. Best validation performance achieved at epoch 197.
After filtering, 2763 genes remain.


In [6]:
# --- Build ms_annotation.h5ad aligned to filtered_ms_adata.h5ad's obs_names ---
# Map prediction indices back to label strings using the celltype category ordering.
celltype_cats = data.obs["celltype"].astype("category").cat.categories.tolist()

import torch as _torch
if isinstance(predictions, _torch.Tensor):
    predictions = predictions.detach().cpu().numpy()

if hasattr(predictions, "argmax"):
    # Predictions may be returned as logits / probabilities.
    if predictions.ndim == 2:
        pred_idx = predictions.argmax(axis=1)
    else:
        pred_idx = predictions
else:
    pred_idx = np.asarray(predictions)

pred_idx = np.asarray(pred_idx).astype(int)
pred_strs = np.array([celltype_cats[i] for i in pred_idx])

# Pick the test rows.
test_mask = data.obs["split"].values == "test"
test_pred_strs = pred_strs[test_mask]
test_celltype_strs = data.obs["celltype"].astype(str).values[test_mask]

# Reload the original test file to get canonical obs_names ordering.
data_test_raw = ad.read_h5ad("/data/benchmark/data/cellPLM/data/filtered_ms_adata.h5ad")
assert len(test_pred_strs) == data_test_raw.n_obs, (
    f"len(predictions on test split)={len(test_pred_strs)} != n_obs={data_test_raw.n_obs}"
)

out = ad.AnnData(np.zeros((data_test_raw.n_obs, 1), dtype=np.float32))
out.obs_names = data_test_raw.obs_names
out.obs["celltype"] = test_celltype_strs
out.obs["predictions"] = pd.Categorical(test_pred_strs)

OUT_PATH = "/data/benchmark/ct_annotation/cellPLM-annotation-wrapper/ms_annotation.h5ad"
out.write_h5ad(OUT_PATH)

acc = accuracy_score(out.obs["celltype"], out.obs["predictions"])
macro_f1 = f1_score(out.obs["celltype"], out.obs["predictions"], average="macro")
print(f"cellPLM  accuracy={acc:.3f}  macro-F1={macro_f1:.3f}  wrote {OUT_PATH}")

cellPLM  accuracy=0.888  macro-F1=0.777  wrote /data/benchmark/ct_annotation/cellPLM-annotation-wrapper/ms_annotation.h5ad
